# Laboratorio 9 — Embeddings

**Natural Language Processing**

En este laboratorio se construyen y comparan embeddings de palabras en español:

- **Corpus principal:** Spanish News Classification (`df_total.csv`), para entrenar embeddings y clasificar noticias.
- **Segundo corpus:** *Don Quijote de la Mancha* (`don-quijote.txt`), para comparar cómo cambia el significado según el texto.
- **Vectores preentrenados:** fastText SUC (tamaño M, 100 dimensiones).

In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gensim
import sklearn
import torch

# Semilla para que los resultados sean reproducibles
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Versiones (se usan para el requirements.txt)
print("gensim", gensim.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)

gensim 4.4.0
scikit-learn 1.8.0
torch 2.14.1


# 1. Preparación de los corpus

### 1.1 Carga de los datos

Cargamos las noticias y el Quijote. Del Quijote quitamos el texto de Project Gutenberg y dejamos solo la novela.

In [ ]:
# Corpus de noticias
df_news = pd.read_csv("df_total.csv")
print("Noticias:", df_news.shape)
print(df_news["Type"].value_counts())

# Corpus del Quijote (utf-8-sig quita el BOM del inicio del archivo)
with open("don-quijote.txt", encoding="utf-8-sig") as f:
    raw_quijote = f.read()

# Nos quedamos solo con la novela: desde el título hasta el aviso final de Gutenberg
inicio = raw_quijote.index("El ingenioso hidalgo don Quijote de la Mancha")
fin = raw_quijote.index("End of Project Gutenberg's Don Quijote")
quijote = raw_quijote[inicio:fin].strip()

print(f"\nQuijote: {len(quijote):,} caracteres")
print(quijote[:300])
print("...")
print(quijote[-200:])

### 1.2 Oraciones y tokens

Separamos cada corpus en oraciones y cada oración en palabras, en orden, porque gensim espera una lista de listas de tokens.

En las noticias falta el espacio después de muchos puntos (`agregó.El`), así que lo agregamos antes de separar las oraciones.

In [5]:
import re
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt_tab", quiet=True)

TOKEN_RE = re.compile(r"\w+")


def segmentar(texto):
    """Divide un texto en oraciones."""
    # Agrega el espacio que falta después de . ! ? cuando sigue una mayúscula (p. ej. "agregó.El")
    texto = re.sub(r"(?<=\w)([.!?])(?=[A-ZÁÉÍÓÚÑ¿¡])", r"\1 ", texto)
    # Une los saltos de línea para no partir oraciones
    texto = re.sub(r"\s+", " ", texto)
    return sent_tokenize(texto, language="spanish")


def tokenizar(oracion):
    """Devuelve los tokens de una oración en orden (sin puntuación)."""
    return TOKEN_RE.findall(oracion)


# Noticias: cada noticia se segmenta por separado
oraciones_news = []
for texto in df_news["news"]:
    for oracion in segmentar(texto):
        tokens = tokenizar(oracion)
        if tokens:  # descarta "oraciones" que eran solo puntuación
            oraciones_news.append(tokens)

# Quijote
oraciones_quijote = [tokenizar(o) for o in segmentar(quijote)]
oraciones_quijote = [t for t in oraciones_quijote if t]

print(f"Noticias: {len(oraciones_news):,} oraciones")
print(f"Quijote:  {len(oraciones_quijote):,} oraciones")
print("\nEjemplo noticias:", oraciones_news[0])
print("Ejemplo Quijote: ", oraciones_quijote[100])

Noticias: 21,805 oraciones
Quijote:  9,513 oraciones

Ejemplo noticias: ['Durante', 'el', 'foro', 'La', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el', 'director', 'de', 'sostenibilidad', 'y', 'clientes', 'globales', 'de', 'BBVA', 'en', 'Colombia', 'Andrés', 'García', 'aseguró', 'que', 'es', 'importante', 'entender', 'que', 'la', 'sostenibilidad', 'no', 'la', 'podemos', 'asociar', 'a', 'mayores', 'costos']
Ejemplo Quijote:  ['No', 'puedo', 'ser', 'tu', 'igual', 'que', 'este', 'decoro', 'se', 'debe', 'a', 'tus', 'proezas', 'y', 'a', 'tu', 'fama', 'puesto', 'que', 'como', 'yo', 'perdiste', 'el', 'seso']


### 1.3 Vocabulario de fastText SUC

Revisamos si SUC distingue mayúsculas y tildes, para normalizar nuestros corpus de la misma forma.

In [6]:
from gensim.models import KeyedVectors

# El archivo pesa 1.1 GB: se guarda fuera del repositorio y no se sube a GitHub
SUC_PATH = "/Users/michelle/Downloads/embeddings-m-model.vec"
suc = KeyedVectors.load_word2vec_format(SUC_PATH, limit=200_000)
print("Vocabulario cargado:", len(suc.key_to_index))

for palabra in ["España", "españa", "espana", "Gobierno", "gobierno", "economía", "economia", "2020", "BBVA", "bbva"]:
    print(f"{palabra:<10}", palabra in suc.key_to_index)

Vocabulario cargado: 200000
España     False
españa     True
espana     True
Gobierno   False
gobierno   True
economía   True
economia   True
2020       True
BBVA       False
bbva       True


### 1.4 Normalización

Según la celda anterior, SUC está en minúsculas, distingue tildes y conserva los números. Normalizamos igual:

- **Minúsculas:** "Gobierno" y "gobierno" son la misma palabra.
- **Tildes:** se conservan, porque cambian el significado ("sé" / "se", "más" / "mas").
- **Números:** se conservan, como en SUC.
- **Puntuación:** ya se quitó al tokenizar.
- **Stopwords y lematización:** no se aplican. Quitar stopwords cambia qué palabras quedan juntas en la ventana de contexto, y lematizar junta "gobiernos" y "gobernó" en una sola forma, perdiendo información de número y tiempo verbal.

In [8]:
def normalizar(tokens):
    """Pasa los tokens a minúsculas."""
    return [t.lower() for t in tokens]


corpus_news = [normalizar(o) for o in oraciones_news]
corpus_quijote = [normalizar(o) for o in oraciones_quijote]

print("Antes:  ", oraciones_news[0][:12])
print("Después:", corpus_news[0][:12])

Antes:   ['Durante', 'el', 'foro', 'La', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el']
Después: ['durante', 'el', 'foro', 'la', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el']


### 1.5 Tamaño de los corpus

En las noticias cada noticia es un documento; en el Quijote, cada capítulo.

In [9]:
def resumen(nombre, n_docs, corpus):
    tokens = [t for oracion in corpus for t in oracion]
    return {"Corpus": nombre, "Documentos": n_docs, "Oraciones": len(corpus),
            "Tokens": len(tokens), "Tipos": len(set(tokens))}


n_capitulos = len(re.findall(r"^Capítulo ", quijote, flags=re.MULTILINE))

tabla_corpus = pd.DataFrame([
    resumen("Noticias", len(df_news), corpus_news),
    resumen("Quijote", n_capitulos, corpus_quijote),
])
tabla_corpus

,Corpus,Documentos,Oraciones,Tokens,Tipos
0,Noticias,1217,21805,637564,29530
1,Quijote,126,9513,381226,22942


### 1.6 Vocabulario según la frecuencia mínima

Solo entran al vocabulario las palabras que aparecen al menos `min_count` veces. Los tokens de las demás quedan fuera.

In [11]:
from collections import Counter

filas = []
for nombre, corpus in [("Noticias", corpus_news), ("Quijote", corpus_quijote)]:
    frecuencias = Counter(t for oracion in corpus for t in oracion)
    total_tokens = sum(frecuencias.values())
    for min_count in [1, 2, 5]:
        vocab = {w for w, c in frecuencias.items() if c >= min_count}
        tokens_fuera = sum(c for w, c in frecuencias.items() if c < min_count)
        filas.append({"Corpus": nombre, "min_count": min_count,
                      "Tamaño del vocabulario": len(vocab),
                      "% tokens fuera": round(100 * tokens_fuera / total_tokens, 2)})

pd.DataFrame(filas)

,Corpus,min_count,Tamaño del vocabulario,% tokens fuera
0,Noticias,1,29530,0.00
1,Noticias,2,17138,1.94
2,Noticias,5,8892,5.39
3,Quijote,1,22942,0.00
4,Quijote,2,11760,2.93
5,Quijote,5,5301,7.38


Con `min_count=2` el vocabulario de noticias baja 42% y solo pierde el 1.9% de los tokens: la mayoría de palabras son muy raras pero suman pocos tokens. Por eso el modelo base usa `min_count=2`.

### 1.7 Pares de skip-gram a mano

Para cada palabra central, sus contextos son las palabras que están hasta 2 posiciones a la izquierda y a la derecha.

In [12]:
def pares_skipgram(oracion, ventana=2):
    pares = []
    for i, centro in enumerate(oracion):
        for j in range(max(0, i - ventana), min(len(oracion), i + ventana + 1)):
            if j != i:
                pares.append((centro, oracion[j]))
    return pares


# Una oración corta de noticias para que se vean todos los pares
oracion_ejemplo = next(o for o in corpus_news if 6 <= len(o) <= 8)
print("Oración:", oracion_ejemplo)

pares = pares_skipgram(oracion_ejemplo, ventana=2)
print(f"\n{len(pares)} pares (palabra central, contexto):")
for centro, contexto in pares:
    print(f"  ({centro}, {contexto})")

Oración: ['viva', 'aerobus', 'es', 'como', 'un', 'hijo', 'para', 'mí']

26 pares (palabra central, contexto):
  (viva, aerobus)
  (viva, es)
  (aerobus, viva)
  (aerobus, es)
  (aerobus, como)
  (es, viva)
  (es, aerobus)
  (es, como)
  (es, un)
  (como, aerobus)
  (como, es)
  (como, un)
  (como, hijo)
  (un, es)
  (un, como)
  (un, hijo)
  (un, para)
  (hijo, como)
  (hijo, un)
  (hijo, para)
  (hijo, mí)
  (para, un)
  (para, hijo)
  (para, mí)
  (mí, hijo)
  (mí, para)


### 1.8 Pares de entrenamiento en todo el corpus de noticias

In [13]:
def contar_pares(corpus, ventana):
    """Cuenta los pares (centro, contexto) sin guardarlos en memoria."""
    total = 0
    for oracion in corpus:
        n = len(oracion)
        for i in range(n):
            total += min(i, ventana) + min(n - 1 - i, ventana)
    return total


n_tokens_news = sum(len(o) for o in corpus_news)
filas = []
for ventana in [2, 5]:
    n_pares = contar_pares(corpus_news, ventana)
    filas.append({"Ventana": ventana, "Tokens": n_tokens_news, "Pares": n_pares,
                  "Pares por token": round(n_pares / n_tokens_news, 2)})

pd.DataFrame(filas)

,Ventana,Tokens,Pares,Pares por token
0,2,637564,2419548,3.79
1,5,637564,5723708,8.98


**¿Por qué word2vec no necesita datos etiquetados?** Porque el propio texto genera las "etiquetas": para cada palabra, la respuesta correcta son las palabras que la rodean. Con solo ~640 mil tokens se obtienen millones de ejemplos de entrenamiento sin que nadie anote nada (aprendizaje auto-supervisado).

# 2. Co-ocurrencia, PPMI y SVD

### 2.1 Matriz de co-ocurrencia

Contamos cuántas veces aparece cada par de palabras a ±2 posiciones, usando las 2,000 palabras más frecuentes de las noticias.

In [ ]:
N_PALABRAS = 2000
VENTANA = 2

frecuencias_news = Counter(t for oracion in corpus_news for t in oracion)
vocab_cooc = [w for w, _ in frecuencias_news.most_common(N_PALABRAS)]
idx_cooc = {w: i for i, w in enumerate(vocab_cooc)}

C = np.zeros((N_PALABRAS, N_PALABRAS), dtype=np.float64)
for oracion in corpus_news:
    ids = [idx_cooc.get(t, -1) for t in oracion]
    for i, wi in enumerate(ids):
        if wi < 0:
            continue
        for j in range(max(0, i - VENTANA), min(len(ids), i + VENTANA + 1)):
            if j != i and ids[j] >= 0:
                C[wi, ids[j]] += 1

print("Forma:", C.shape)
print(f"Ceros: {100 * (C == 0).mean():.2f}%")

In [ ]:
# Dispersión de la BoW reportada en el Laboratorio 2 (mismas noticias)
comparacion = pd.DataFrame([
    {"Matriz": "Co-ocurrencia (2,000 x 2,000)", "% ceros": round(100 * (C == 0).mean(), 2)},
    {"Matriz": "BoW Lab 2 - 1,000 palabras (1,217 x 1,000)", "% ceros": 89.03},
    {"Matriz": "BoW Lab 2 - 5,000 palabras (1,217 x 5,000)", "% ceros": 96.73},
    {"Matriz": "BoW Lab 2 - vocabulario completo (1,217 x 21,911)", "% ceros": 99.14},
])
comparacion